In [1]:
import pandas as pd
import numpy as np

print("🔄 Loading Bank Credit Card Data...")

# Load the raw banking dataset
df_credit = pd.read_csv("../data/raw/creditcard.csv")

print(f"✅ Data loaded successfully. Shape: {df_credit.shape[0]} rows, {df_credit.shape[1]} columns\n")

print("--- 🔍 Checking for Missing Values ---")
missing_count = df_credit.isnull().sum().sum()
print(f"Total missing values: {missing_count}")

print("\n--- 👥 Checking for Duplicate Rows ---")
duplicate_count = df_credit.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

# Drop duplicates to keep the dataset pristine
if duplicate_count > 0:
    print("🔄 Removing duplicate rows...")
    df_credit = df_credit.drop_duplicates().reset_index(drop=True)
    print(f"✅ New Shape: {df_credit.shape[0]} rows, {df_credit.shape[1]} columns")

🔄 Loading Bank Credit Card Data...
✅ Data loaded successfully. Shape: 284807 rows, 31 columns

--- 🔍 Checking for Missing Values ---
Total missing values: 0

--- 👥 Checking for Duplicate Rows ---
Number of duplicate rows found: 1081
🔄 Removing duplicate rows...
✅ New Shape: 283726 rows, 31 columns


In [2]:
print("=== 💳 Bank Credit Card Class Imbalance ===")
counts = df_credit['Class'].value_counts()
percentages = df_credit['Class'].value_counts(normalize=True) * 100

for cls in counts.index:
    label = "Fraudulent" if cls == 1 else "Legitimate"
    print(f"{label} (Class {cls}): {counts[cls]} records ({percentages[cls]:.3f}%)")

=== 💳 Bank Credit Card Class Imbalance ===
Legitimate (Class 0): 283253 records (99.833%)
Fraudulent (Class 1): 473 records (0.167%)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("🔄 Separating features and target variable...")
X_bank = df_credit.drop(columns=['Class'])
y_bank = df_credit['Class']

print("🔄 Performing stratified train-test split (80/20)...")
X_bank_train, X_bank_test, y_bank_train, y_bank_test = train_test_split(
    X_bank, y_bank, test_size=0.2, stratify=y_bank, random_state=42
)

print("🔄 Scaling 'Time' and 'Amount' columns...")
scaler_bank = StandardScaler()

# Copy dataframes to safe variables
X_bank_train_scaled = X_bank_train.copy()
X_bank_test_scaled = X_bank_test.copy()

# Fit only on the training set features to avoid data leakage
columns_to_scale = ['Time', 'Amount']
X_bank_train_scaled[columns_to_scale] = scaler_bank.fit_transform(X_bank_train[columns_to_scale])
X_bank_test_scaled[columns_to_scale] = scaler_bank.transform(X_bank_test[columns_to_scale])

print("✅ Features normalized successfully.")

🔄 Separating features and target variable...
🔄 Performing stratified train-test split (80/20)...
🔄 Scaling 'Time' and 'Amount' columns...
✅ Features normalized successfully.


In [4]:
from imblearn.over_sampling import SMOTE

print("📊 --- Class Distribution BEFORE Banking SMOTE ---")
print(f"Legitimate (Class 0): {np.bincount(y_bank_train)[0]}")
print(f"Fraudulent (Class 1): {np.bincount(y_bank_train)[1]}")

print("\n🔄 Resampling banking minority class via SMOTE...")
smote_bank = SMOTE(random_state=42)
X_bank_train_resampled, y_bank_train_resampled = smote_bank.fit_resample(X_bank_train_scaled, y_bank_train)

print("\n📊 --- Class Distribution AFTER Banking SMOTE ---")
print(f"Legitimate (Class 0): {np.bincount(y_bank_train_resampled)[0]}")
print(f"Fraudulent (Class 1): {np.bincount(y_bank_train_resampled)[1]}")

📊 --- Class Distribution BEFORE Banking SMOTE ---
Legitimate (Class 0): 226602
Fraudulent (Class 1): 378

🔄 Resampling banking minority class via SMOTE...

📊 --- Class Distribution AFTER Banking SMOTE ---
Legitimate (Class 0): 226602
Fraudulent (Class 1): 226602


In [5]:
print("💾 Saving completely processed bank training/testing sets...")

np.savez_compressed('../data/processed/creditcard_train_test.npz', 
                    X_train=X_bank_train_resampled, 
                    X_test=X_bank_test_scaled.values, 
                    y_train=y_bank_train_resampled, 
                    y_test=y_bank_test.values)

print("🎉 Success! All Banking stream components for Task 1 are officially finalized and stored.")

💾 Saving completely processed bank training/testing sets...
🎉 Success! All Banking stream components for Task 1 are officially finalized and stored.
